In [1]:
import os, json, re, time
import pandas as pd
from openai import OpenAI
import numpy as np
import pandas as pd
from datetime import datetime

In [2]:
client = OpenAI(
    api_key= "YOUR_API_KEY",
    base_url="https://api.deepseek.com"
)

In [3]:
QUESTIONS = [
    "1.您认为该思维片段多大可能是【抑郁症患者】说的。[1:完全不是; 9:绝对是]",
    "2.您认为该思维片段多大程度上是反映说话者【内部感受】的（躯体，内心感觉）。[1:完全没有; 9:几乎都是]",
    "3.您认为该思维片段多大程度上是【内部导向】的（独立于外部刺激，自发产生的想法/画面）。[1:完全没有; 9:几乎都是]",
    "4.您认为该思维片段多大程度上是【外部导向】的（由外界刺激诱发的想法，例如噪声、设备、实验任务等）。[1:完全没有; 9:几乎都是]",
    "5.您认为该思维片段多大程度上是关注【自我】的。[1:完全没有; 9:几乎都是]",
    "6.您认为该思维片段多大程度上是关注【他人】的。[1:完全没有; 9:几乎都是]",
    "7.您认为该思维片段多大程度上说的是【正性的、积极】的事情。[1:完全没有; 9:几乎都是]",
    "8.您认为该思维片段多大程度上说的是【负性的、消极】的事情。[1:完全没有; 9:几乎都是]",
    "9.您认为该思维片段涉及【问题解决】的程度。（试图解决某个问题，或思考如何达到某个目标）。[1:完全没有; 9:几乎都是]",
    "10.您认为该思维片段反映说话者【自责、内疚】的程度。[1:完全没有; 9:几乎都是]",
    "11.您认为该思维片段反映说话者【担忧】的程度。[1:完全没有; 9:几乎都是]",
    "12.您认为该思维片段反映说话者【生气】的程度。[1:完全没有; 9:几乎都是]",
    "13.您认为该思维片段反映说话者【厌恶】的程度。[1:完全没有; 9:几乎都是]",
    "14.您认为该思维片段反映说话者【恐惧】的程度。[1:完全没有; 9:几乎都是]",
    "15.您认为该思维片段反映说话者【悲伤】的程度。[1:完全没有; 9:几乎都是]",
    "16.您认为该思维片段反映说话者【惊喜】的程度。[1:完全没有; 9:几乎都是]",
    "17.您认为该思维片段反映说话者【幸福】的程度。[1:完全没有; 9:几乎都是]",
    "18.您认为该思维片段反映说话者【喜欢】的程度。[1:完全没有; 9:几乎都是]",
    "19.您认为该思维片段反映说话者【愉悦】的程度。[1:完全没有; 9:几乎都是]",
    "20.您认为该思维片段反映说话者【自卑】的程度。[1:完全没有; 9:几乎都是]",
    "21.您认为该思维片段反映说话者【失望】的程度。[1:完全没有; 9:几乎都是]",
    "22.您认为该思维片段反映说话者【反省】的程度。[1:完全没有; 9:几乎都是]",
    "23.您认为该思维片段多大程度上是关注【过去】的事情。[1:完全没有; 9:几乎都是]",
    "24.您认为该思维片段多大程度上是关注【未来】的事情。[1:完全没有; 9:几乎都是]",
    "25.您认为该思维片段多大程度上是关注【当下】的事情。[1:完全没有; 9:几乎都是]",
    "26.您认为该思维片段反映说话者【反刍】的程度。（反刍：对负性情绪本身及其原因和可能后果的反复思考。）[1:完全没有; 9:几乎都是]",
]


In [4]:
QUESTION_IDS = list(range(1, 27))

SYSTEM_PROMPT = """你是一个严格的心理学专家，请根据下面的问题，给出你对该思维片段的评分。
必须输出JSON对象，键为q1...q26，值为1-9整数。
禁止输出任何解释或额外文本。"""

Q_BLOCK = "\n".join([f"Q{i}: {q}" for i, q in zip(QUESTION_IDS, QUESTIONS)])

def _extract_json(raw):
    m = re.search(r"\{.*\}", raw, flags=re.S)
    if not m:
        raise ValueError("No JSON found")
    return json.loads(m.group(0))

def rate_26dims(text, retries=3):
    if not isinstance(text, str) or text.strip() == "":
        return {f"q{i}": None for i in QUESTION_IDS}

    user_prompt = f"""请按Q1-Q26评分：

{Q_BLOCK}

文本：{text}

只输出JSON，如：
{{"q1":4,"q2":3,...,"q26":5}}"""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="deepseek-v4-flash",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                stream=False,
                temperature=0,
                response_format={"type": "json_object"}
            )

            data = _extract_json(response.choices[0].message.content)

            return {f"q{i}": int(data.get(f"q{i}", None))
                    if str(data.get(f"q{i}", "")).isdigit()
                    else None
                    for i in QUESTION_IDS}

        except Exception as e:
            print("Retry:", e)
            time.sleep(2 ** attempt)

    return {f"q{i}": None for i in QUESTION_IDS}

In [ ]:

file_path = "D:/2025THU/SELF_OTHER/LLMRating_Task/Raw/AllRowData.xlsx"
df = pd.read_excel(file_path, engine="openpyxl")

In [ ]:
scores_series = df["Response"].apply(rate_26dims)
score_df = pd.DataFrame(list(scores_series))
df_out = pd.concat([df, score_df], axis=1)
df_out.to_excel("DP_V4_flash_all.xlsx", index=False)

In [8]:
QCOLS = [f"q{i}" for i in range(1, 27)]

def _row_has_missing_any_q(row) -> bool:
    """
    判定该行 q1..q26 是否存在缺失：
    - None / NaN
    - 或者不在 1..9 的值（保险起见）
    """
    vals = row[QCOLS]
    # pandas 会把 None 变成 NaN，所以用 isna 覆盖两者
    if vals.isna().any():
        return True
    # 如果有奇怪的值（比如 0, 10, 字符串等），也当作缺失需要重评
    for v in vals.values:
        try:
            iv = int(v)
        except Exception:
            return True
        if iv < 1 or iv > 9:
            return True
    return False

def rerate_rows_with_missing(
    df_out: pd.DataFrame,
    text_col: str = "Response",
    max_rounds_per_row: int = 2,
    sleep_between_rounds: float = 0.0
):
    """
    找到任意 q1..q26 缺失的行 -> 对该行完整重评 -> 写回 df_out[q1..q26]
    同时返回：miss 行索引列表、以及重评日志 DataFrame
    """
    # 1) 找到需要重评的行索引
    need_idx = df_out.index[df_out.apply(_row_has_missing_any_q, axis=1)].tolist()
    print(f"Rows needing rerate: {len(need_idx)}")

    logs = []

    # 2) 逐行重评并写回
    for idx in need_idx:
        text = df_out.at[idx, text_col]
        if not isinstance(text, str) or text.strip() == "":
            # 空文本不重评也没意义，记录一下
            logs.append({
                "index": idx,
                "status": "skip_empty_text",
                "rounds_used": 0,
                "time_utc": datetime.utcnow().isoformat()
            })
            continue

        success = False
        rounds_used = 0

        for r in range(1, max_rounds_per_row + 1):
            rounds_used = r
            new_scores = rate_26dims(text)  # 你已有的函数

            # 检查本次是否全齐（没有 None）
            if all(new_scores.get(c) in range(1, 10) for c in QCOLS):
                # 写回 df_out
                for c in QCOLS:
                    df_out.at[idx, c] = new_scores[c]
                success = True
                break

            if sleep_between_rounds:
                time.sleep(sleep_between_rounds)

        logs.append({
            "index": idx,
            "status": "rerate_ok" if success else "rerate_failed",
            "rounds_used": rounds_used,
            "time_utc": datetime.utcnow().isoformat()
        })

    log_df = pd.DataFrame(logs)
    return need_idx, log_df, df_out


In [9]:
# ====== 调用 ======
missing_row_indices, rerate_log, df_out = rerate_rows_with_missing(
    df_out,
    text_col="Response",
    max_rounds_per_row=3,     # 每行最多重评3次（你可改）
    sleep_between_rounds=0.0  # 可设 0.5 防止太快触发限流
)

print("Missing row indices:", missing_row_indices[:20], "..." if len(missing_row_indices) > 20 else "")
print(rerate_log["status"].value_counts())


Rows needing rerate: 7


C:\Users\axnw683\AppData\Local\Temp\ipykernel_14200\482086853.py:74: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "time_utc": datetime.utcnow().isoformat()


Missing row indices: [6086, 7063, 7303, 7478, 8476, 9716, 9723] 
status
rerate_ok    7
Name: count, dtype: int64


In [11]:
# ====== 保存结果（可选）=====
df_out.to_excel("C:/rating_task/AllRowData_test_scored_rerated.xlsx", index=False)
#rerate_log.to_excel("D:/2025THU/SELF_OTHER/LLMRating_Task/rerate_log.xlsx", index=False)